# Notebook with the pipeline as steps

## 1. Files segmentation and statements extraction

- python run_segmentation.py --model gemini-2-5-flash --parallel 20 --seg_batch_words 600
- python run_extraction.py --model gemini-2-5-flash --parallel 20 --seg_batch_words 600

In [3]:
import os
from pathlib import Path
from src.run_segmentation import xml_to_json, segment_all_jsons

xml_to_json('input_data')
output_dir = "output_data/segmented_files"
if not Path(output_dir).exists():
    os.makedirs(output_dir)
segment_all_jsons('input_data', output_dir, 1000, "gpt-5.2", 8)

In [ ]:
from src.run_extraction import run_extraction

run_extraction("output_data/segmented_files", "output_data/statements.csv", 600, "gemini-2.5-flash", 8)

## 2. Classification

- python run_classification.py --model gemini-2-5-flash --parallel 20 --seg_batch_words 600

In [ ]:
from src.run_classification import classify_all_statements, match_and_replace_statements
import pandas as pd
import ast, os
merge_stm = False
input_data = "output_data/statements.csv"
output_data = "output_data/clasified_statements.csv"

if merge_stm == True:
    statements = pd.read_csv(input_data, header=0)
    directory = os.path.dirname(output_data) 

    statements['original_paragraph'] = statements['paragraph'].apply(
        lambda x: ast.literal_eval(x)['original_paragraph']
    )
    gold = pd.read_csv("input_data/gold_statements.csv", header=0)
    merged = match_and_replace_statements(statements, gold)
    merged.to_csv(directory + '/merged_statements.csv', index=False)
else:
    merged = pd.read_csv(input_data, header=0)
    
classify_all_statements(merged, output_data, 20, "gemini-2.5-flash", 8)

## 3. Negation of statements

- python run_classification.py --model gemini-2-5-flash --parallel 20 --seg_batch_words 600

In [ ]:
from src.run_negation import run_negation

statements = "output_data/clasified_statements.csv"
new_statements_df =run_negation("output_data/clasified_statements.csv", 20, "accounts/fireworks/models/kimi-k2p5", 8)
statements = pd.concat([statements, new_statements_df], ignore_index=True)
statements.to_csv("output_data/negation_statements.csv", index=False)

## 4. Verification

- python run_veracity.py --model gemini-2-5-flash --parallel 20 

In [ ]:
from src.run_veracity import run_veracity, make_output_path, evaluation_verdict, evaluation_verdict_by_gold_class

challenge_df = pd.read_csv("output_data/negation_statements.csv", header=0)
judged = run_veracity(challenge_df, "accounts/fireworks/models/kimi-k2p5", 8)
output_path = make_output_path("accounts/fireworks/models/kimi-k2p5", "output_data/judged_statements")
output_path.parent.mkdir(parents=True, exist_ok=True)
judged.to_csv(output_path, index=False)

In [ ]:
eval_df = judged.dropna(subset=["Gold Verification", "veredict"]).copy()
eval_df["Gold Verification"] = eval_df["Gold Verification"].astype(bool)
eval_df["veredict"] = eval_df["veredict"].astype(bool)
if len(eval_df) > 0:
    evaluation_verdict(eval_df)
    if "Gold Class" in eval_df.columns:
        evaluation_verdict_by_gold_class(eval_df)